# 챕터 4 — 품목의 진입·퇴출: 20년간 무엇이 생기고 사라졌나 (2007–2025)

한국 수출 품목(HS6)이 얼마나 새로 등장하고 사라졌는지를 인과 해석 없이 서술하는 기술통계 노트북이다. **개수 지표**라 HS 개정을 `dim_hs6_concordance`로 이어야 정확하다(안 그러면 개정 착시). 함께 있는 문서 `chapter04.md`가 이 결과를 이야기로 엮은 것이다.

## 이 데이터베이스에 대하여 (출처·집계 기준)

이 데이터베이스는 관세청이 OpenAPI(품목별 국가별 수출입실적(GW), https://www.data.go.kr/data/15100475/openapi.do)로 공개하는 월별 수출입 통계를 **2007년 1월부터 2026년 3월까지** 한데 모아 하나의 파일로 만든 것이다.

**수치의 집계 기준(관세청 정의).** 수출입 신고 통관 자료를 국가 및 HS Code(2·4·6·10단위)별로 집계한 국가별 품목별 무역통계다. 금액은 미화(USD)이며, 수출은 FOB(신고금액), 수입은 CIF(과세가격) 기준이다. 중량은 순중량(kg). 국가는 수출은 최종목적국, 수입은 원산국을 원칙으로 하며 무역통계부호상 ISO 코드로 분류한다. 단순 통과물품이나 일시 반입·반출 물품은 제외된다(물적 자원의 증감이 없으므로). 통계는 매월 수출입 신고의 정정·취하를 반영해 전월까지 자료를 현행화한다(주기 1개월).

## 0. 규칙과 방법

- **개수 지표**: 거래액이 아니라 품목 종수. 챕터 1의 봉인 2가 걸린다(마지막 절).
- **안정 코드**: 시기별 판본(2007–11→2007, 2012–16→2012, 2017–21→2017, 2022–→그대로)을 HS2022 기준으로 잇는다. 미매칭(연도당 0–5종)은 제외.
- **완전연도만**(2007–2025), 수출액>0 HS6.
- **경계 절단**: 진입·퇴출은 2007·2025 두 시점의 집합 차이로만 정의(그 이전/이후는 알 수 없음).

In [ ]:
import os
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB 없음: {DB_PATH} — Releases에서 받아 data/processed/ 에 배치")
con = duckdb.connect(DB_PATH, read_only=True)
def q(sql): return con.sql(sql).df()

# 각 (연도, hs6)를 HS2022 안정코드로 잇는 공통 CTE
BASE = '''
WITH ym AS (SELECT yyyymm//100 yr, SUBSTR(hs10,1,6) hs6 FROM fact_trade
            WHERE yyyymm//100 BETWEEN 2007 AND 2025 AND exp_dlr>0 GROUP BY 1,2),
ver AS (SELECT yr, hs6, CASE WHEN yr BETWEEN 2007 AND 2011 THEN '2007'
             WHEN yr BETWEEN 2012 AND 2016 THEN '2012'
             WHEN yr BETWEEN 2017 AND 2021 THEN '2017' ELSE '2022' END hv FROM ym),
mapped AS (SELECT v.yr, v.hs6, CASE WHEN v.hv='2022' THEN v.hs6 ELSE c.hs2022 END scode
           FROM ver v LEFT JOIN dim_hs6_concordance c
                ON v.hv<>'2022' AND c.past_version=v.hv AND c.hs_past=v.hs6)
'''

# 금액 비중·상위 품목용 CTE (진입=2025에만, 퇴출=2007에만)
SET2 = '''
WITH set25 AS (SELECT DISTINCT SUBSTR(hs10,1,6) h FROM fact_trade WHERE yyyymm//100=2025 AND exp_dlr>0),
stable07 AS (SELECT DISTINCT c.hs2022 scode
             FROM (SELECT DISTINCT SUBSTR(hs10,1,6) h FROM fact_trade WHERE yyyymm//100=2007 AND exp_dlr>0) a
             JOIN dim_hs6_concordance c ON c.past_version='2007' AND c.hs_past=a.h),
e25 AS (SELECT SUBSTR(hs10,1,6) hs6, SUM(exp_dlr) v FROM fact_trade WHERE yyyymm//100=2025 AND exp_dlr>0 GROUP BY 1),
e07 AS (SELECT SUBSTR(hs10,1,6) hs6, SUM(exp_dlr) v FROM fact_trade WHERE yyyymm//100=2007 AND exp_dlr>0 GROUP BY 1),
nm AS (SELECT SUBSTR(hs10,1,6) hs6, ANY_VALUE(name_ko) pn FROM dim_hs10 WHERE name_ko IS NOT NULL GROUP BY 1)
'''
print("연결 완료 —", f"{con.sql('SELECT COUNT(*) FROM fact_trade').fetchone()[0]:,}", "거래행")

## 3. 개정 착시 — 코드만 세면 틀린다

In [ ]:
# (a) 번호 그대로: 2007 vs 2025 진입·퇴출
raw = q('''
    WITH a AS (SELECT DISTINCT SUBSTR(hs10,1,6) h FROM fact_trade WHERE yyyymm//100=2007 AND exp_dlr>0),
         b AS (SELECT DISTINCT SUBSTR(hs10,1,6) h FROM fact_trade WHERE yyyymm//100=2025 AND exp_dlr>0)
    SELECT (SELECT COUNT(*) FROM b WHERE h NOT IN (SELECT h FROM a)) AS 진입,
           (SELECT COUNT(*) FROM a WHERE h NOT IN (SELECT h FROM b)) AS 퇴출
''')
# (b) 개정 연결: 안정코드 기준
cc = q(BASE+''',
    s07 AS (SELECT DISTINCT scode FROM mapped WHERE yr=2007 AND scode IS NOT NULL),
    s25 AS (SELECT DISTINCT scode FROM mapped WHERE yr=2025 AND scode IS NOT NULL)
    SELECT (SELECT COUNT(*) FROM s25 WHERE scode NOT IN (SELECT scode FROM s07)) AS 진입,
           (SELECT COUNT(*) FROM s07 WHERE scode NOT IN (SELECT scode FROM s25)) AS 퇴출,
           (SELECT COUNT(*) FROM s07 WHERE scode IN (SELECT scode FROM s25)) AS 지속
''')
raw_e, raw_x = int(raw['진입'][0]), int(raw['퇴출'][0])
c_e, c_x, persist = int(cc['진입'][0]), int(cc['퇴출'][0]), int(cc['지속'][0])
print(f"번호 그대로:  진입 {raw_e}, 퇴출 {raw_x}")
print(f"개정 연결:    진입 {c_e}, 퇴출 {c_x}, 지속 {persist}")

x=np.arange(2); w=0.38
f=plt.figure(figsize=(7,3.8)); ax=f.gca()
ax.bar(x-w/2, [raw_e,raw_x], w, label='Raw code (revision illusion)', color='lightgray')
ax.bar(x+w/2, [c_e,c_x], w, label='Concordance-corrected', color='steelblue')
for i,(rv,cv) in enumerate(zip([raw_e,raw_x],[c_e,c_x])):
    ax.text(i-w/2, rv+8, str(rv), ha='center', fontsize=8); ax.text(i+w/2, cv+8, str(cv), ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(['Entry (new HS6)','Exit (gone HS6)'])
ax.set_ylabel('Number of HS6 codes'); ax.set_title('Product Churn 2007 to 2025: raw vs concordance')
ax.legend(fontsize=7); ax.grid(alpha=.3, axis='y'); f.tight_layout(); plt.show()

## 4. 실제 구도 — 품목은 놀랍도록 안정적

In [ ]:
# 지속/진입/퇴출 구성 (개정 연결 기준)
f=plt.figure(figsize=(7,2.6)); ax=f.gca()
cats=['Persist (both)','Entered (2025 only)','Exited (2007 only)']; vals=[persist,c_e,c_x]
ax.barh(cats[::-1], vals[::-1], color=['#4c78a8','seagreen','crimson'][::-1])
for i,v in enumerate(vals[::-1]): ax.text(v+30, i, str(v), va='center', fontsize=8)
ax.set_xlabel('Number of stable HS6 codes'); ax.set_xlim(0,5200)
ax.set_title('2007 vs 2025 Export Products (concordance-based)'); f.tight_layout(); plt.show()
print(f"2007년 품목의 {round(100*persist/(persist+c_x),1)}%가 2025년에도 지속")

In [ ]:
# 다양성 추이: 번호 그대로 vs 안정코드
var = q(BASE+'''SELECT yr, COUNT(DISTINCT hs6) raw_hs6, COUNT(DISTINCT scode) stable_hs6
               FROM mapped GROUP BY yr ORDER BY yr''')
print(var.to_string(index=False))
f=plt.figure(figsize=(8,3.6)); ax=f.gca()
ax.plot(var['yr'], var['stable_hs6'], marker='o', ms=3, label='Stable (concordance)')
ax.plot(var['yr'], var['raw_hs6'], marker='s', ms=3, ls='--', color='gray', label='Raw code')
ax.set_ylabel('Distinct HS6 exported'); ax.set_xlabel('Year'); ax.set_ylim(0,5600)
ax.set_title('Export Product Variety, Korea (2007-2025)')
ax.legend(fontsize=7); ax.grid(alpha=.3); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
f.tight_layout(); plt.show()

## 5. 금액으로 보면 — 진입·퇴출의 비중과 상위 품목

In [ ]:
# 진입·퇴출 품목의 금액 비중
ent = q(SET2+'''
    SELECT ROUND(SUM(v) FILTER (WHERE hs6 NOT IN (SELECT scode FROM stable07))/1e8,0) 억,
           ROUND(SUM(v)/1e8,0) 전체,
           ROUND(100.0*SUM(v) FILTER (WHERE hs6 NOT IN (SELECT scode FROM stable07))/SUM(v),3) pct
    FROM e25''')
ext = q(SET2+''', gone AS (SELECT e07.hs6,e07.v FROM e07 WHERE NOT EXISTS
          (SELECT 1 FROM dim_hs6_concordance c WHERE c.past_version='2007' AND c.hs_past=e07.hs6
                 AND c.hs2022 IN (SELECT h FROM set25)))
    SELECT ROUND((SELECT SUM(v) FROM gone)/1e8,0) 억, ROUND(SUM(v)/1e8,0) 전체,
           ROUND(100.0*(SELECT SUM(v) FROM gone)/SUM(v),3) pct FROM e07''')
print(f"진입 품목: {ent['억'][0]:.0f}억 / {ent['전체'][0]:.0f}억 = {ent['pct'][0]}%  (2025 수출 기준)")
print(f"퇴출 품목: {ext['억'][0]:.0f}억 / {ext['전체'][0]:.0f}억 = {ext['pct'][0]}%  (2007 수출 기준)")

In [ ]:
# 금액 기준 상위 진입 품목 (2025)
q(SET2+'''
    SELECT e25.hs6 AS HS6, nm.pn AS 품목명, ROUND(e25.v/1e8,1) AS 수출2025_억
    FROM e25 LEFT JOIN nm ON nm.hs6=e25.hs6
    WHERE e25.hs6 NOT IN (SELECT scode FROM stable07)
    ORDER BY e25.v DESC LIMIT 10''')

In [ ]:
# 금액 기준 상위 퇴출 품목 (2007). 2007 전용 코드는 폐지코드라 품목명이 불완전할 수 있음.
q(SET2+''', gone AS (SELECT e07.hs6,e07.v FROM e07 WHERE NOT EXISTS
          (SELECT 1 FROM dim_hs6_concordance c WHERE c.past_version='2007' AND c.hs_past=e07.hs6
                 AND c.hs2022 IN (SELECT h FROM set25)))
    SELECT gone.hs6 AS HS6, nm.pn AS 품목명_2026기준, ROUND(gone.v/1e8,1) AS 수출2007_억
    FROM gone LEFT JOIN nm ON nm.hs6=gone.hs6 ORDER BY gone.v DESC LIMIT 10''')

## 마무리 (한계)

번호만 세면 진입 941·퇴출 408처럼 보이지만, 개정을 이으면 실제는 **진입 327·퇴출 326**이고 **4,696종(약 93%)이 20년간 지속**한다. 품목 구성은 놀랍도록 안정적이다. 한계(개수 지표라 특히 중요): 봉인 2(다대다 증식·완전 사각지대 124/63/34), 좌·우 절단(진입·퇴출은 두 시점 집합 차이일 뿐), 미매칭 제외, 수출 활동 기준. 상세는 `chapter04.md` 참조.

In [ ]:
con.close()
print("연결 종료.")